# Interrogatorio de Datos — Titanic**Minería de Datos — Semana / Unidad correspondiente**

In [ ]:
import pandas as pdimport numpy as npurl = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"df = pd.read_csv(url)df.head()

(891, 12)

## Pregunta A — Tasa de supervivencia global

In [ ]:
df['Survived'].value_counts(normalize=True) * 100

0    61.6161621    38.383838Name: Survived, dtype: float64

**Respuesta A:** La tasa de supervivencia global fue de **38.38%**; el 61.62% de los pasajeros murió.

## Pregunta B — Supervivencia por género

In [ ]:
df.groupby('Sex')['Survived'].mean() * 100

Sexfemale    74.203822male      18.890815Name: Survived, dtype: float64

**Respuesta B:** Sobrevivió el **74.20%** de las mujeres contra solo el **18.89%** de los hombres. Los datos confirman matemáticamente el código "mujeres y niños primero".

## Pregunta C — Outliers en Fare (IQR)

In [ ]:
q1, q3 = df['Fare'].quantile([0.25, 0.75])iqr = q3 - q1limite_superior = q3 + 1.5 * iqroutliers = df[df['Fare'] > limite_superior]print(len(outliers))print(outliers['Pclass'].value_counts())

116Pclass1    1043      72      5Name: Pclass, dtype: int64

**Respuesta C:** Se detectaron **116 outliers** por encima del límite superior del IQR. La gran mayoría (**104 de 116**) pertenecen a **1ra clase**, lo cual confirma que son tarifas de pasajeros de lujo, no errores de captura.

## Pregunta D — Media vs. Mediana de Fare

In [ ]:
df['Fare'].mean()

32.204207968574636

In [ ]:
df['Fare'].median()

14.4542

**Respuesta D:** La media (**32.20**) es más del doble que la mediana (**14.45**), lo cual indica una **distribución muy sesgada a la derecha (skew positivo)**: pocos pasajeros pagaron tarifas extremadamente altas y jalan el promedio hacia arriba, mientras la mayoría pagó tarifas bajas.Si esta variable se usa sin escalar en un algoritmo basado en distancias euclidianas (KNN), **`Fare` dominará por completo el cálculo de distancia** solo por tener una escala numérica mucho mayor que columnas como `Age` o `Pclass` — el modelo terminaría "pensando" que la tarifa es la única característica relevante para decidir qué tan parecidos son dos pasajeros.

## Pregunta E — Muestreo estratificado

In [ ]:
from sklearn.model_selection import train_test_splitmuestra, _ = train_test_split(    df, train_size=150, stratify=df['Survived'], random_state=42)muestra['Survived'].value_counts(normalize=True) * 100

0    61.3333331    38.666667Name: proportion, dtype: float64

**Respuesta E:** Se usa `stratify=df['Survived']` para forzar que la muestra de 150 pasajeros respete la misma proporción de sobrevivientes (~38.4%) que tiene el dataset completo (61.33% / 38.67% en la muestra vs. 61.62% / 38.38% real). Esto evita el **sesgo de muestreo (sampling bias)**: sin estratificar, el azar podría generar una muestra con muy pocos o demasiados sobrevivientes, distorsionando el entrenamiento del modelo hacia la clase mayoritaria.

## Pregunta F — Sesgo por NaN en Age

In [ ]:
df.groupby('Survived')['Age'].mean()

Survived0    30.6261791    28.343690Name: Age, dtype: float64

In [ ]:
df[df['Age'].isnull()]['Pclass'].value_counts()

Pclass3    1361     302     11Name: count, dtype: int64

**Respuesta F:** De los 177 valores nulos en `Age`, 136 (la mayoría) pertenecen a pasajeros de **3ra clase**, y 125 corresponden a pasajeros que **murieron**. Como `.mean()` descarta silenciosamente los NaN, el promedio de edad de los "no sobrevivientes" se calcula solo con el subconjunto de pasajeros de 3ra clase que sí tenían edad registrada — probablemente no representativo del resto. Esto introduce un **sesgo de datos faltantes no aleatorios (MNAR)**: el promedio resultante no refleja la verdadera distribución de edad de los fallecidos de 3ra clase, y cualquier inferencia basada en él hereda esa distorsión.

## Pregunta G — ¿Eliminar los outliers de Fare?

In [ ]:
outliers[['Name','Fare','Pclass']].sort_values('Fare', ascending=False).head(5)

                                   Name      Fare  Pclass258                    Ward, Miss. Anna  512.3292       1737              Lesurer, Mr. Gustave J  512.3292       1679  Cardeza, Mr. Thomas Drake Martinez  512.3292       127       Fortune, Mr. Charles Alexander  263.0000       1341      Fortune, Miss. Alice Elizabeth  263.0000       1

**Respuesta G:** No deberían eliminarse con `.drop()`. Los boletos de 512 libras pertenecen a pasajeros reales de 1ra clase (ej. la familia Cardeza, documentada históricamente como una de las más ricas a bordo). Esto es **señal, no ruido**: representa un segmento real del mercado (suites de lujo), no un error de captura. La estrategia correcta es **transformar** la variable (log-transform o escalamiento robusto) para que no domine el modelo, en vez de descartar información real.

## Pregunta H — Imputación de Age por título extraído de Name

In [ ]:
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')df.groupby('Title')['Age'].median().sort_values(ascending=False).head(6)

TitleCapt     70.0Col      58.0Sir      49.0Major    48.5Lady     48.0Rev      46.5Name: Age, dtype: float64

**Respuesta H:** Es superior porque el título captura información real correlacionada con la edad que la mediana global ignora — "Master" identifica niños varones (mediana baja), "Mr"/"Mrs" adultos, "Sir"/"Major"/"Capt" casi siempre adultos mayores. Imputar con la mediana global le asignaría, por ejemplo, la edad de un adulto promedio a un niño con título "Master", introduciendo error sistemático. Agrupar por título reduce la varianza del error de imputación al usar subgrupos mucho más homogéneos que la población completa.

## Pregunta I — Varianza de Survived

In [ ]:
df['Survived'].var()

0.23677221654749742

**Respuesta I:** Si la varianza fuera exactamente 0.0, significaría que **todos los pasajeros tienen el mismo valor** en `Survived` (todos murieron o todos sobrevivieron) — no existe variabilidad que explicar. Un algoritmo de clasificación no podría entrenarse: no hay señal que separar, ya que la variable de respuesta no tiene clases distintas. La mayoría de los modelos fallarían al entrenar, o generarían un error, porque no hay nada que optimizar.

## Pregunta J — Agrupación triple y maldición de la dimensionalidad

In [ ]:
df.groupby(['Pclass','Sex','Embarked'])['PassengerId'].count()

Pclass  Sex     Embarked1       female  C            43                Q             1                S            48        male    C            42                Q             1                S            792       female  C             7                Q             2                S            67        male    C            10                Q             1                S            973       female  C            23                Q            33                S            88        male    C            43                Q            39                S           265Name: PassengerId, dtype: int64

**Respuesta J:** Ocurrirá **sobreajuste (overfitting)**. Si el modelo aprende reglas basadas en grupos de 1-2 pasajeros, está memorizando casos particulares (ruido) en vez de capturar un patrón generalizable. Esas reglas funcionarán perfecto sobre esos casos exactos de entrenamiento, pero fallarán al generalizar a pasajeros nuevos que no encajen exactamente en esos micro-grupos.